# SegmentEveryGrain — río Mana (SfM, 1 mm)

Adaptación del cuaderno original de Zoltán Sylvester para una **tesela piloto** del ortomosaico del río Mana, con JPEG + worldfile y salida georreferenciada EPSG:32619.

**No es todavía un procesamiento por lotes.** La calidad de segmentación y las métricas requieren validación de campo.


<a target="_blank" href="https://colab.research.google.com/github/zsylvester/segmenteverygrain/blob/main/notebooks/Segment_every_grain_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Install and import packages

### Set up interactive figure backend in Colab

In [ ]:
# this is needed to make figures in Colab interactive
!pip install ipympl
exit(0) # this restarts the runtime after installing ipympl -- otherwise you get an error when switching the matplotlib backend to ipympl

In [ ]:
from google.colab import output

output.enable_custom_widget_manager()

In [ ]:
%matplotlib ipympl

### Install the other dependencies

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
import sys

!{sys.executable} -m pip install opencv-python matplotlib
!{sys.executable} -m pip install sam2
!{sys.executable} -m pip install segmenteverygrain
!{sys.executable} -m pip install rtree
!wget https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam_checkpoint = "sam2.1_hiera_large.pt"
device = "cuda"

sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import measure
from importlib import reload
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from tqdm import trange, tqdm

## Download Unet model weights and create Unet model

In [ ]:
!wget "https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt"

In [ ]:
# U-Net model (PyTorch)
unet = seg.load_unet("seg_model_smooth_labels.pt")

## Run segmentation

Grains are supposed to be well defined in the image; e.g., if a grain consists of only a few pxiels, it is unlikely to be detected.

The segmentation can take a few minutes even for medium-sized images, so do not start with large images (downsample them if necessary).

In [ ]:
# Configuración del ortomosaico del río Mana (JPEG + worldfile)
from google.colab import drive
from pathlib import Path
from affine import Affine
from PIL import Image

drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/gravas_mana')  # AJUSTAR
JPG = CARPETA / 'odm_orthophoto.jpg'                  # AJUSTAR
WLD = CARPETA / 'odm_orthophoto_jpg.wld'              # AJUSTAR
SALIDA = CARPETA / 'resultados_seg'
SALIDA.mkdir(parents=True, exist_ok=True)
EPSG = 'EPSG:32619'  # Confirmado por el autor
assert JPG.is_file(), f'No existe el JPEG: {JPG}'
assert WLD.is_file(), f'No existe el worldfile: {WLD}'
A, D, B, E, C, F = map(float, WLD.read_text().split())
transform = Affine(A, B, C-(A+B)/2, D, E, F-(D+E)/2)
px_per_m = 1 / ((abs(A) + abs(E)) / 2)
print('Transformación:', transform)
print('Píxeles por metro:', px_per_m)
# Para este primer ensayo, no se intenta abrir la imagen completa como ndarray.
Image.MAX_IMAGE_PIXELS = None
with Image.open(JPG) as im:
    ancho, alto = im.size
print('Dimensiones:', ancho, alto)


### Seleccionar una tesela de prueba

Se conserva la resolución nativa. Las coordenadas `X0`, `Y0` son índices de píxel desde la esquina superior izquierda. El worldfile describe los **centros** de los píxeles; la transformación `Affine` representa sus esquinas. Para el análisis completo se implementará teselado con solapamiento y eliminación de duplicados.


In [ ]:
# Seleccionar una tesela con gravas visibles; modificar tras inspección.
X0, Y0 = 10000, 8000
TAM = 1024
assert 0 <= X0 < ancho and 0 <= Y0 < alto
assert X0 + TAM <= ancho and Y0 + TAM <= alto
with Image.open(JPG) as im:
    image = np.asarray(im.crop((X0, Y0, X0+TAM, Y0+TAM)).convert('RGB'))
transform_tesela = transform * Affine.translation(X0, Y0)
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title(f'Tesela ({X0}, {Y0}), {TAM} × {TAM} píxeles')
plt.axis('off')
plt.show()


In [ ]:
%%time
# Ejecutar después de seleccionar la tesela y cargar U-Net y SAM.
image_pred = seg.predict_image(image, unet, I=256)
labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=20.0)
all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
    sam, image, image_pred, coords, labels,
    min_area=400.0, plot_image=True,
    remove_edge_grains=False,
    remove_large_objects=False,
)
print('Objetos segmentados:', len(all_grains))


## Interactive editing of segmentation results

**Note:** Interactive editing in Colab may be slower than in a local Jupyter notebook due to network latency.

Convert the polygons to Grain objects and use the interactive `GrainPlot` interface:

**Mouse controls:**
- **Left-click** on existing grain: Select/unselect
- **Left-click** in grain-free area: Instant grain creation
- **Alt + Left-click**: Multi-prompt grain creation
- **Alt + Right-click**: Background prompt for multi-prompt creation
- **Shift + drag**: Draw scale bar for unit conversion
- **Ctrl** (hold): Temporarily hide grain masks

**Keyboard controls:**
- **d**: Delete selected grains
- **m**: Merge selected grains
- **z**: Undo last created grain
- **Esc**: Clear selection and prompts

In [ ]:
# Convert polygons to Grain objects
grains = si.polygons_to_grains(all_grains, image=image)
for g in tqdm(grains, desc='Measuring grains'):
    g.measure()

In [ ]:
# Edición interactiva OPCIONAL: ejecutar solo si se desea corregir máscaras.
# Si se omite esta celda, continuar con el resumen y exportación.
predictor = SAM2ImagePredictor(sam)
predictor.set_image(image)
plot = si.GrainPlot(
    grains, image=image, predictor=predictor,
    blit=False, color_palette='tab20b', figsize=(12, 8),
    scale_m=0.1, color_by=None, px_per_m=px_per_m
)
plot.activate()


**Solo si activaste el editor interactivo:** ejecuta la siguiente celda cuando termines. Si no lo activaste, sáltala.


In [ ]:
# Retrieve updated grains and deactivate interactive mode
grains = plot.get_grains()
plot.deactivate()

## Get grain size distribution

La escala se calcula directamente desde el worldfile (en metros por píxel). No es necesario dibujar una barra de escala ni activar el editor interactivo.


In [ ]:
# La escala procede del worldfile; no depende de activar GrainPlot.
summary = si.get_summary(grains, px_per_m)
hist = seg.plot_histogram_of_axis_lengths(
    summary['major_axis_length'] * 1000,
    summary['minor_axis_length'] * 1000,
    binsize=0.25,
)
summary.head()


In [ ]:
# Guardar medidas y geometrías en coordenadas locales de la tesela.
# La exportación NO depende del editor interactivo.
import json
from shapely.geometry import shape, mapping
from shapely.ops import transform as transform_geom

prefijo = SALIDA / f'mana_x{X0}_y{Y0}_{TAM}'
si.save_grains(str(prefijo) + '_pixel.geojson', grains)
summary = si.save_summary(str(prefijo) + '_summary.csv', grains,
                          px_per_m=px_per_m)
si.save_histogram(str(prefijo) + '_histograma.jpg', summary=summary)
si.save_mask(str(prefijo) + '_mask.png', grains, image, scale=False)

# Transformar geometrías del GeoJSON (coordenadas de píxel) a UTM.
# Se comprueba que las coordenadas sean locales a la tesela.
ruta_pixel = Path(str(prefijo) + '_pixel.geojson')
geo = json.loads(ruta_pixel.read_text())
features = geo['features']
if features:
    bounds = [shape(f['geometry']).bounds for f in features if f.get('geometry')]
    if bounds:
        mnx = min(b[0] for b in bounds); mxx = max(b[2] for b in bounds)
        mny = min(b[1] for b in bounds); mxy = max(b[3] for b in bounds)
        assert -2 <= mnx <= TAM+2 and -2 <= mxx <= TAM+2, 'Coordenadas no locales: revisar exportación'
        assert -2 <= mny <= TAM+2 and -2 <= mxy <= TAM+2, 'Coordenadas no locales: revisar exportación'

def pixel_a_utm(x, y, z=None):
    import numpy as np
    x = np.asarray(x); y = np.asarray(y)
    xx = transform_tesela.a*x + transform_tesela.b*y + transform_tesela.c
    yy = transform_tesela.d*x + transform_tesela.e*y + transform_tesela.f
    return (xx, yy) if z is None else (xx, yy, z)

for f in features:
    if f.get('geometry'):
        f['geometry'] = mapping(transform_geom(pixel_a_utm, shape(f['geometry'])))
geo['crs'] = {'type': 'name', 'properties': {'name': EPSG}}
ruta_utm = Path(str(prefijo) + '_utm19n.geojson')
ruta_utm.write_text(json.dumps(geo))
print('Exportados:', ruta_pixel, ruta_utm, sep='\n')
print('Clastos:', len(features), '| EPSG:', EPSG)
summary.head()
